In [ ]:
from ultralytics import YOLO

model = YOLO('../weights/hazard/yolo26s.pt')

results = model('../dataset/images/photo4.jpg')

print(results[0])

In [ ]:
import sys
from pathlib import Path
from types import ModuleType

try:
    import pkg_resources
except ModuleNotFoundError:
    from packaging.version import parse as parse_version

    pkg_resources = ModuleType('pkg_resources')
    pkg_resources.parse_version = parse_version
    sys.modules['pkg_resources'] = pkg_resources

from huggingface_hub.errors import RepositoryNotFoundError

if 'huggingface_hub.utils._errors' not in sys.modules:
    _errors = ModuleType('huggingface_hub.utils._errors')
    _errors.RepositoryNotFoundError = RepositoryNotFoundError
    sys.modules['huggingface_hub.utils._errors'] = _errors

import torch
import torch.serialization as _torch_serialization

# YOLOv5 checkpoints need pickle. Always wrap the real torch.serialization.load,
# so a previous recursive monkey-patch (from re-running this cell) is replaced.
if not getattr(torch.load, '_yolov5_weights_only_patch', False):
    def _torch_load(*args, _orig=_torch_serialization.load, **kwargs):
        kwargs.setdefault('weights_only', False)
        return _orig(*args, **kwargs)

    _torch_load._yolov5_weights_only_patch = True
    torch.load = _torch_load

import yolov5

_yolov5_root = str(Path(yolov5.__file__).resolve().parent)
if _yolov5_root not in sys.path:
    sys.path.insert(0, _yolov5_root)

from IPython.display import display
from PIL import Image, ImageDraw

model = yolov5.load('uisikdag/yolo-v5-construction-machine-detection')
model.conf = 0.25
model.iou = 0.45

image_path = '../dataset/images/photo7.png'
results = model(image_path, size=640)
results.print()

detections = results.pandas().xyxy[0]
print('classes:', model.names)
print(detections)

image = Image.open(image_path).convert('RGB')
draw = ImageDraw.Draw(image)
for row in detections.itertuples():
    draw.rectangle([row.xmin, row.ymin, row.xmax, row.ymax], outline='red', width=3)
    draw.text((row.xmin, max(row.ymin - 14, 0)), f'{row.name} {row.confidence:.2f}', fill='red')

display(image)

In [ ]:
model = YOLO('../weights/construction-vehicle/best.pt')

results = model('../dataset/images/photo4.jpg')

print(results[0])

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

classes = [
    "crawler excavator", "wheeled excavator", "mini excavator",
    "bulldozer", "wheel loader", "backhoe loader", "skid steer loader",
    "telescopic handler", "tower crane", "mobile crane", "crawler crane",
    "dump truck", "concrete mixer truck", "motor grader", "road roller",
    "asphalt paver", "compactor", "forklift", "scraper"
]

st = SentenceTransformer('sentence-transformers/clip-ViT-B-32')
emb = st.encode(classes, normalize_embeddings=True, convert_to_tensor=True)

model = YOLO("../weights/yolo-world/yolov8x-worldv2.pt") 
model.model.txt_feats = emb.unsqueeze(0)
model.model.model[-1].nc = len(classes)
model.model.names = classes

if getattr(model, "predictor", None):
    model.predictor.model.names = classes

results = model.predict("../dataset/images/photo7.png", conf=0.25)

In [ ]:
import cv2
annotated_image = results[0].plot()

cv2.imwrite('my_result_image.jpg', annotated_image)